In [7]:
import pandas as pd
import numpy as np
import sqlite3
import os

df = pd.read_csv("../data/supply_chain_data.csv")
print(df.shape)
df.head()

(100, 24)


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [8]:
print(df.columns.tolist())
print("\n", df.dtypes)
print("\n", df.isnull().sum())

['Product type', 'SKU', 'Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Customer demographics', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location', 'Lead time', 'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Inspection results', 'Defect rates', 'Transportation modes', 'Routes', 'Costs']

 Product type                object
SKU                         object
Price                      float64
Availability                 int64
Number of products sold      int64
Revenue generated          float64
Customer demographics       object
Stock levels                 int64
Lead times                   int64
Order quantities             int64
Shipping times               int64
Shipping carriers           object
Shipping costs             float64
Supplier name               object
Location                    object
Lead time                    int64
Production vol

In [9]:
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

df["total_lead_time"] = df["lead_times"] + df["manufacturing_lead_time"] + df["shipping_times"]
df["cost_per_unit"] = df["costs"] / df["order_quantities"].replace(0, np.nan)
df["revenue_per_unit"] = df["revenue_generated"] / df["number_of_products_sold"].replace(0, np.nan)
df["stockout_risk"] = (df["stock_levels"] < df["order_quantities"] * 0.3).astype(int)
df["defect_flag"] = (df["defect_rates"] > df["defect_rates"].quantile(0.75)).astype(int)

print(df[["sku","total_lead_time","cost_per_unit","stockout_risk","defect_flag"]].head(8))

    sku  total_lead_time  cost_per_unit  stockout_risk  defect_flag
0  SKU0               40       1.955751              0            0
1  SKU1               62      13.596367              0            1
2  SKU2               39       1.612730              1            1
3  SKU3               37       4.318240              0            1
4  SKU4               14      16.490011              1            0
5  SKU5               47       3.567594              0            0
6  SKU6               47       2.316709              1            0
7  SKU7               19      72.914210              0            0


In [10]:
revenue_sorted = df.sort_values("revenue_generated", ascending=False).copy()
revenue_sorted["cumulative_pct"] = revenue_sorted["revenue_generated"].cumsum() / revenue_sorted["revenue_generated"].sum() * 100

def abc_class(pct):
    if pct <= 70:
        return "A"
    elif pct <= 90:
        return "B"
    return "C"

revenue_sorted["abc_class"] = revenue_sorted["cumulative_pct"].apply(abc_class)
df = df.merge(revenue_sorted[["sku", "abc_class"]], on="sku", how="left")

os.makedirs("../data", exist_ok=True)
conn = sqlite3.connect("../data/supply_chain.db")
df.to_sql("supply_chain", conn, if_exists="replace", index=False)

verify = pd.read_sql("SELECT abc_class, COUNT(*) as sku_count FROM supply_chain GROUP BY abc_class", conn)
print(verify)
conn.close()

  abc_class  sku_count
0         A         49
1         B         24
2         C         27


In [11]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/supply_chain_data.csv")

conn = sqlite3.connect("../data/supply_chain.db")

df.to_sql("supply_chain_data", conn, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM supply_chain_data LIMIT 5", conn)

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [12]:
df = pd.read_csv("../data/supply_chain_data.csv")

df.columns = df.columns.str.lower().str.replace(" ", "_")

conn = sqlite3.connect("../data/supply_chain.db")

df.to_sql("supply_chain_data", conn, if_exists="replace", index=False)

100